# ClustMRF on TREC Deep Learning 2023 / MS MARCO v2.1

**Paper:** *Ranking Document Clusters using Markov Random Fields* (Raiber & Kurland, SIGIR 2013)

**Corpus:** MS MARCO v2.1 documents (~11.6M docs; loaded as a judgment-pool corpus)

**Topics / Qrels:** TREC DL 2023 — 82 evaluated queries, ~16K qrels

**Metrics:** MAP@50 · P@5 · NDCG@5 · NDCG@10 · NDCG@100

---

## Algorithm Overview

ClustMRF ranks query-specific **document clusters** using the MRF framework (Eq. 3 of paper):

```
score(C, Q) = Σ_{l∈L(G)} λ_l · f_l(l)
```

**Three clique types** (Fig. 1):

| Clique | Feature | Description |
|--------|---------|-------------|
| lQD (Q + one doc) | `geo-qsim` | (1/\|C\|) log sim(Q,d) → geometric mean of query sims |
| lQC (Q + all docs) | `min/max/stdv-qsim` | Statistics of {sim(Q,d)}\_{d∈C} |
| lC (docs only) | `geo/min/max-{dsim,entropy,icompress}` | Query-independent doc measures |

**Clustering:** Each d ∈ D_init is center of C(d) = {d} ∪ {k−1 nearest neighbours by TF-IDF cosine sim}.

**Initial list:** The paper uses MRF/SDM (Metzler & Croft 2005) with λ_T=0.85, λ_O=0.10, λ_U=0.05.

**Document ranking from cluster ranking (§4.1):** Clusters sorted best→worst; constituent docs appended in sim(Q,d) order, skipping duplicates.

## 1  Environment Setup

In [15]:
import os, sys, warnings, pathlib, logging
warnings.filterwarnings('ignore')

# Java 21 (Temurin) — required by PyTerrier / Terrier.
# Install: brew install --cask temurin@21
os.environ.setdefault(
    'JAVA_HOME',
    '/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home'
)

# Add project root to path so we can import src.*
ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

import ir_datasets
import ir_measures
from ir_measures import MAP, nDCG, P

import pyterrier as pt
# PyTerrier 1.x modern init — avoids deprecated pt.init() wrapper that
# appended 'm' to the mem string, producing an invalid -Xmx flag.
if not pt.java.started():
    pt.java.set_memory_limit(4096)   # 4 GB in MB
    pt.java.init()

logging.basicConfig(level=logging.WARNING)   # quiet noisy loggers

print(f'PyTerrier  : {pt.__version__}')
print(f'ir_datasets: {ir_datasets.__version__}')
print(f'ir_measures: {ir_measures.__version__}')

PyTerrier  : 1.0.4
ir_datasets: 0.5.11
ir_measures: 0.4.3


## 2  Dataset: TREC DL 2023 / MS MARCO v2.1

**Corpus strategy:** The full MS MARCO v2.1 corpus is ~11.6M documents across 60 sharded `.json.gz` files inside `data/trec-rag-2024/corpus/msmarco_v2.1_doc.tar`.  Indexing all 11.6M documents is unnecessary for a re-ranking experiment.  Instead we build a **judgment-pool corpus**:

- All ~15K documents that appear in the DL 2023 qrels (so every judged document is retrieved-able)
- A reservoir sample of 1 000 background documents per shard (× 60 shards = 60K) to give BM25 realistic competition

Total indexed corpus: ~75K documents.  The index is built once and cached.

In [16]:
DATA_DIR   = ROOT / 'data' / 'trec-rag-2024'
TAR_PATH   = DATA_DIR / 'corpus' / 'msmarco_v2.1_doc.tar'
TOPICS_PATH = DATA_DIR / 'topics' / 'topics.dl23.txt'
QRELS_PATH  = DATA_DIR / 'qrels' / 'qrels.dl23-doc-msmarco-v2.1.txt'
INDEX_DIR   = ROOT / 'data' / 'indexes' / 'trec-dl23-pool'

for p in [TAR_PATH, TOPICS_PATH, QRELS_PATH]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {p.relative_to(ROOT)}')

print(f'\nIndex dir  : {INDEX_DIR.relative_to(ROOT)}')

✓  data/trec-rag-2024/corpus/msmarco_v2.1_doc.tar
✓  data/trec-rag-2024/topics/topics.dl23.txt
✓  data/trec-rag-2024/qrels/qrels.dl23-doc-msmarco-v2.1.txt

Index dir  : data/indexes/trec-dl23-pool


## 3  Topics and Qrels

In [17]:
# ── Topics (700 in file, filter to the 82 with qrels) ────────────────────────
all_topics = {}
with open(TOPICS_PATH) as f:
    for line in f:
        parts = line.strip().split('\t', 1)
        if len(parts) == 2:
            all_topics[parts[0]] = parts[1]

print(f'Topics in file : {len(all_topics)}')

# ── Qrels (standard TREC format: qid 0 docid rel) ────────────────────────────
qrels_rows = []
qrel_docids = set()

with open(QRELS_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 4:
            qid, _, docid, rel = parts
            qrels_rows.append({'query_id': qid, 'doc_id': docid, 'relevance': int(rel)})
            qrel_docids.add(docid)

qrels_df = pd.DataFrame(qrels_rows)
judged_qids = set(qrels_df['query_id'].unique())

# Filter topics to only judged queries
topics_df = pd.DataFrame([
    {'qid': qid, 'query': all_topics[qid]}
    for qid in judged_qids if qid in all_topics
]).sort_values('qid').reset_index(drop=True)

print(f'Judged queries : {len(topics_df)}')
print(f'Qrel rows      : {len(qrels_df):,}')
print(f'Unique doc IDs : {len(qrel_docids):,}')
print(f'\nRelevance distribution:')
print(qrels_df['relevance'].value_counts().sort_index().to_string())
topics_df.head()

Topics in file : 700
Judged queries : 82
Qrel rows      : 15,995
Unique doc IDs : 15,588

Relevance distribution:
relevance
0    10560
1     2679
2     1447
3     1309


,qid,query
0,2001010,cost comparison of funerals in australia
1,2001459,dog age by teeth
2,2001575,fda definition of verification
3,2002075,how do you clean smoke off walls
4,2002168,how does a bounty hunter make money


## 4  Build Index (Judgment-Pool Corpus)

Streams all 60 shards from the tar.  For each shard:
- Keeps every document whose `docid` appears in the qrels
- Reservoir-samples 1 000 background documents for BM25 competition

**First run:** ~8–12 min (streams 29 GB, builds ~75K-doc Terrier index).
**Subsequent runs:** loads cached index in seconds.

In [18]:
import tarfile, gzip, json, random, shutil
from tqdm.auto import tqdm

INDEX_DIR.mkdir(parents=True, exist_ok=True)
props_file   = INDEX_DIR / 'data.properties'
blocks_marker = INDEX_DIR / '.blocks_enabled'   # sentinel: index built with blocks=True

N_BACKGROUND_PER_SHARD = 1000

def corpus_iter():
    """
    Stream judgment-pool corpus from the tar.
    Yields every qrel-referenced doc + a reservoir sample of background docs per shard.
    """
    rng = random.Random(42)
    with tarfile.open(str(TAR_PATH)) as tar:
        members = sorted(
            [m for m in tar.getmembers() if m.name.endswith('.json.gz')],
            key=lambda m: m.name,
        )
        for member in tqdm(members, desc='Streaming shards', unit='shard'):
            f = tar.extractfile(member)
            if f is None:
                continue
            qrel_buf, reservoir, n_seen = [], [], 0
            with gzip.open(f) as gz:
                for raw in gz:
                    doc   = json.loads(raw)
                    docid = doc['docid']
                    text  = ' '.join(filter(None, [doc.get('title',''), doc.get('body','')]))
                    rec   = {'docno': docid, 'text': text}
                    if docid in qrel_docids:
                        qrel_buf.append(rec)
                    else:
                        n_seen += 1
                        if len(reservoir) < N_BACKGROUND_PER_SHARD:
                            reservoir.append(rec)
                        else:
                            j = rng.randrange(n_seen)
                            if j < N_BACKGROUND_PER_SHARD:
                                reservoir[j] = rec
            yield from qrel_buf
            yield from reservoir


# SDM proximity operators require block (position) information.
# Rebuild the index if it was created without blocks.
if props_file.exists() and not blocks_marker.exists():
    print('Existing index lacks block info (needed for SDM). Removing and rebuilding…')
    shutil.rmtree(str(INDEX_DIR))
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

if not props_file.exists():
    print(f'Building judgment-pool index (with blocks) from {TAR_PATH.name} …')
    print('This streams 29 GB — expect ~10–15 min on first run.')
    indexer = pt.IterDictIndexer(
        str(INDEX_DIR),
        overwrite  = True,
        meta       = {'docno': 40, 'text': 8192},
        text_attrs = ['text'],
        blocks     = True,          # ← position info for SDM #1() and #uw() operators
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
    )
    indexer.index(corpus_iter())
    blocks_marker.touch()
    print('Index built.')
else:
    print(f'Index (with blocks) exists at {INDEX_DIR} — loading.')

index = pt.IndexFactory.of(str(props_file))
stats = index.getCollectionStatistics()
print(f'Documents    : {stats.numberOfDocuments:,}')
print(f'Tokens       : {stats.numberOfTokens:,}')
print(f'Unique terms : {stats.numberOfUniqueTerms:,}')

Existing index lacks block info (needed for SDM). Removing and rebuilding…
Building judgment-pool index (with blocks) from msmarco_v2.1_doc.tar …
This streams 29 GB — expect ~10–15 min on first run.


Streaming shards:  13%|█▎        | 8/60 [01:37<09:38, 11.13s/shard]

20:46:36.788 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (msmarco_v2.1_doc_08_1668540108) - further warnings are suppressed


Streaming shards: 100%|██████████| 60/60 [13:25<00:00, 13.43s/shard]


20:58:40.472 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 1 empty documents
Index built.
Documents    : 75,588
Tokens       : 78,050,493
Unique terms : 827,685


## 5  Initial Retrieval

The paper uses **MRF / SDM** (Sequential Dependence Model, Metzler & Croft 2005) as the initial retrieval:

```
score_SDM(Q, d) = λ_T · Σ_t log P(t|d)  +  λ_O · Σ_{t1,t2} log P(#1(t1,t2)|d)  +  λ_U · Σ_{t1,t2} log P(#uw8(t1,t2)|d)
                  λ_T=0.85                   λ_O=0.10                                  λ_U=0.05
```

Proximity operators `#1()` (ordered, gap≤1) and `#uw8()` (unordered, window 8) require block (position) data.
The index was rebuilt with `blocks=True` so full SDM proximity scoring is active.

We also retain a **BM25** run as a standalone baseline for comparison.

In [19]:
# ── BM25 baseline ────────────────────────────────────────────────────────────
bm25_br = pt.BatchRetrieve(
    index,
    wmodel      = 'BM25',
    num_results = 100,
    metadata    = ['docno', 'text'],
    controls    = {'bm25.b': 0.75, 'bm25.k_1': 1.2},
    verbose     = True,
)

print('Running BM25 on judged queries…')
bm25_run = bm25_br.transform(topics_df)
print(f'BM25: {len(bm25_run):,} rows  ({bm25_run.qid.nunique()} queries)')

# ── SDM initial retrieval (Metzler & Croft 2005) ─────────────────────────────
# Terrier's DependenceModelPreProcess defaults: λ_T=0.85, λ_O=0.10, λ_U=0.05
# These match the paper's parameters exactly — no explicit weight configuration needed.
# DirichletLM with μ=2500 (Zhai & Lafferty 2001 standard value).
sdm_rewrite  = pt.rewrite.SDM()
dirichlet_br = pt.BatchRetrieve(
    index,
    wmodel      = 'DirichletLM',
    num_results = 100,
    metadata    = ['docno', 'text'],
    controls    = {'c': 2500},
    verbose     = True,
)
sdm_pipe = sdm_rewrite >> dirichlet_br

print('\nRunning SDM + DirichletLM on judged queries…')
sdm_run = sdm_pipe.transform(topics_df)
print(f'SDM : {len(sdm_run):,} rows  ({sdm_run.qid.nunique()} queries)')
sdm_run.head()

Running BM25 on judged queries…


TerrierRetr(BM25): 100%|██████████| 82/82 [00:03<00:00, 22.85q/s]


BM25: 8,067 rows  (82 queries)

Running SDM + DirichletLM on judged queries…


TerrierRetr(DirichletLM): 100%|██████████| 82/82 [00:05<00:00, 14.92q/s]


SDM : 8,067 rows  (82 queries)


,qid,docid,docno,text,rank,score,query,query_0
0,2001010,50883,msmarco_v2.1_doc_40_508796200,How much does a funeral cost in Australia? 202...,0,15.547732,cost comparison funerals australia #combine:0=...,cost comparison of funerals in australia
1,2001010,43327,msmarco_v2.1_doc_34_366736729,How Much Does The Average Funeral Cost? | Best...,1,15.034520,cost comparison funerals australia #combine:0=...,cost comparison of funerals in australia
2,2001010,57035,msmarco_v2.1_doc_45_860497921,2021 Breakdown of Average Funeral Costs 2021 B...,2,14.986152,cost comparison funerals australia #combine:0=...,cost comparison of funerals in australia
3,2001010,8064,msmarco_v2.1_doc_06_589500388,Cremation Services | Prices & Costs of a Crema...,3,14.860986,cost comparison funerals australia #combine:0=...,cost comparison of funerals in australia
4,2001010,24876,msmarco_v2.1_doc_19_1407307860,FUNERALS MADE EASY | Get Quotes | eziFunerals ...,4,14.400061,cost comparison funerals australia #combine:0=...,cost comparison of funerals in australia


## 6  Baseline Evaluation (BM25 & SDM)

In [20]:
from ir_measures import MAP, nDCG, P

MEASURES = [MAP @ 50, P @ 5, nDCG @ 5, nDCG @ 10, nDCG @ 100]

bm25_eval = bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
bm25_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, bm25_eval)

sdm_eval = sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
sdm_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, sdm_eval)

print('Baselines — TREC DL 2023 (82 queries, judgment-pool corpus)')
print(f'{"Measure":<20} {"BM25":>10} {"SDM":>10}')
print('=' * 42)
for m in MEASURES:
    print(f'  {str(m):<18} {float(bm25_agg[m]):>10.4f} {float(sdm_agg[m]):>10.4f}')

Baselines — TREC DL 2023 (82 queries, judgment-pool corpus)
Measure                    BM25        SDM
  AP@50                  0.2722     0.2641
  P@5                    0.5707     0.5976
  nDCG@5                 0.4703     0.5018
  nDCG@10                0.4676     0.4747
  nDCG@100               0.5672     0.5655


## 7  ClustMRF

The `ClustMRF` class lives in `src/algorithms/clust_mrf.py`.

**Feature functions** (all in log space; add-ε = 1e-10 before log):

| Clique | Feature | Default weight |
|--------|---------|---------------|
| lQD | `geo-qsim` = (1/k) Σ log sim(Q,d) | 0.132 |
| lQC | `stdv-qsim` = log stddev({sim}) | 0.143 |
| lQC | `max-qsim` = log max sim(Q,d) | 0.121 |
| lQC | `min-qsim` = log min sim(Q,d) | 0.088 |
| lC  | `min-dsim` = log min P_dsim(d) | 0.110 |
| lC  | `min-icompress` = log min P_icompress(d) | 0.099 |
| lC  | `max-dsim` | 0.066 |
| lC  | `geo-icompress` | 0.077 |
| lC  | `geo-dsim` | 0.055 |
| lC  | `max-icompress` | 0.044 |
| lC  | `geo-entropy` | 0.033 |
| lC  | `min-entropy` | 0.022 |
| lC  | `max-entropy` | 0.011 |

Where: **P_dsim(d)** = avg cosine sim of d to other cluster members · **P_entropy(d)** = term entropy · **P_icompress(d)** = len(d)/len(gzip(d))

Weights are proportional to the feature importance order in Table 3 (non-web setting) of the paper.
In the paper, weights are learned with SVM^rank on a train set.

In [ ]:
from src.algorithms.clust_mrf import ClustMRF

clustmrf = ClustMRF(
    index  = index,
    k      = 5,      # cluster size: doc + 4 nearest neighbours
    n_docs = 250,    # |D_init|
    # Feature weights: defaults proportional to paper's Table 3 importance order
    # (non-web setting; sw1/sw2/pr/spam features not included)
    # Set all equal (1/13 ≈ 0.077) to weight features uniformly, or tune via CV.
)

total_w = sum([
    clustmrf.w_geo_qsim, clustmrf.w_stdv_qsim, clustmrf.w_max_qsim, clustmrf.w_min_qsim,
    clustmrf.w_min_dsim, clustmrf.w_max_dsim, clustmrf.w_geo_dsim,
    clustmrf.w_min_icompress, clustmrf.w_geo_icompress, clustmrf.w_max_icompress,
    clustmrf.w_geo_entropy, clustmrf.w_min_entropy, clustmrf.w_max_entropy,
])
print(f'ClustMRF ready  k={clustmrf.k}  n_docs={clustmrf.n_docs}')
print(f'Total feature weight: {total_w:.3f}  (should be ≈ 1.0)')

ClustMRF ready  k=5  n_docs=100
Total feature weight: 1.001  (should be ≈ 1.0)


## 8  ClustMRF Retrieval

In [ ]:
import time

_clustmrf_path = RUNS_DIR / 'clustmrf.txt'
if _clustmrf_path.exists():
    print('Loading cached ClustMRF run…')
    clustmrf_run = load_trec_run(_clustmrf_path)
else:
    t0 = time.time()
    clustmrf_run = clustmrf.transform(sdm_run)
    elapsed = time.time() - t0
    print(f'Done in {elapsed:.1f}s  ({elapsed / sdm_run["qid"].nunique():.2f}s per query)')
    save_trec_run(clustmrf_run, _clustmrf_path, 'ClustMRF_SDM')

print(f'Total rows : {len(clustmrf_run):,}')
clustmrf_run.head()

## 9  Evaluate ClustMRF

In [23]:
clustmrf_eval = clustmrf_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
clustmrf_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, clustmrf_eval)

print('ClustMRF — TREC DL 2023 (82 queries)')
print('=' * 42)
for m in MEASURES:
    print(f'  {str(m):<15} {float(clustmrf_agg[m]):.4f}')

ClustMRF — TREC DL 2023 (82 queries)
  AP@50           0.2628
  P@5             0.5585
  nDCG@5          0.4553
  nDCG@10         0.4598
  nDCG@100        0.5619


## 10  Results Comparison

In [24]:
# ── Aggregate comparison table ────────────────────────────────────────────────
rows = []
for name, agg in [('BM25', bm25_agg), ('SDM (init)', sdm_agg), ('ClustMRF (SDM)', clustmrf_agg)]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    rows.append(row)

results_table = pd.DataFrame(rows).set_index('System')

# Δ rows: SDM gain over BM25, ClustMRF gain over SDM
results_table.loc['Δ (SDM − BM25)']          = results_table.loc['SDM (init)']    - results_table.loc['BM25']
results_table.loc['Δ (ClustMRF − SDM init)'] = results_table.loc['ClustMRF (SDM)'] - results_table.loc['SDM (init)']

print('\n=== TREC DL 2023 Results (82 queries) ===')
print(results_table.to_string())


=== TREC DL 2023 Results (82 queries) ===
                          AP@50     P@5  nDCG@5  nDCG@10  nDCG@100
System                                                            
BM25                     0.2722  0.5707  0.4703   0.4676    0.5672
SDM (init)               0.2641  0.5976  0.5018   0.4747    0.5655
ClustMRF (SDM)           0.2628  0.5585  0.4553   0.4598    0.5619
Δ (SDM − BM25)          -0.0081  0.0269  0.0315   0.0071   -0.0017
Δ (ClustMRF − SDM init) -0.0013 -0.0391 -0.0465  -0.0149   -0.0036


In [ ]:
RUNS_DIR = ROOT / 'data' / 'runs' / 'trec-dl23'
RUNS_DIR.mkdir(parents=True, exist_ok=True)


def save_trec_run(run_df: pd.DataFrame, path: pathlib.Path, tag: str) -> None:
    with open(path, 'w') as f:
        for qid, group in run_df.sort_values(['qid', 'rank']).groupby('qid'):
            for rank, row in enumerate(group.itertuples(), start=1):
                f.write(f'{qid} Q0 {row.docno} {rank} {row.score:.6f} {tag}\n')
    print(f'Saved  → {path.relative_to(ROOT)}')


def load_trec_run(path: pathlib.Path) -> pd.DataFrame:
    """Load a TREC run file into a DataFrame with qid / docno / rank / score columns.
    Sufficient for evaluation and score interpolation — no text/query columns."""
    rows = []
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 5:
                rows.append({'qid': p[0], 'docno': p[2],
                             'rank': int(p[3]), 'score': float(p[4])})
    return pd.DataFrame(rows)

In [25]:
# ── Per-query MAP: SDM vs ClustMRF (the fair comparison from the paper) ───────
sdm_perq = ir_measures.iter_calc([MAP], qrels_df, sdm_eval)
clustmrf_perq = ir_measures.iter_calc([MAP], qrels_df, clustmrf_eval)

sdm_map_q      = {r.query_id: r.value for r in sdm_perq      if r.measure == MAP}
clustmrf_map_q = {r.query_id: r.value for r in clustmrf_perq  if r.measure == MAP}

perq_rows = [
    {
        'qid'          : qid,
        'SDM_MAP'      : round(sdm_map_q.get(qid, 0.0), 4),
        'ClustMRF_MAP' : round(clustmrf_map_q.get(qid, 0.0), 4),
    }
    for qid in sorted(sdm_map_q)
]
perq_df = pd.DataFrame(perq_rows)
perq_df['Δ MAP'] = (perq_df['ClustMRF_MAP'] - perq_df['SDM_MAP']).round(4)

wins   = (perq_df['Δ MAP'] > 0).sum()
losses = (perq_df['Δ MAP'] < 0).sum()
ties   = (perq_df['Δ MAP'] == 0).sum()

print(f'ClustMRF vs SDM per-query MAP:')
print(f'  Wins   : {wins}')
print(f'  Losses : {losses}')
print(f'  Ties   : {ties}')
print()
print('Top-5 gains:')
print(perq_df.nlargest(5, 'Δ MAP')[['qid','SDM_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))
print()
print('Top-5 losses:')
print(perq_df.nsmallest(5, 'Δ MAP')[['qid','SDM_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))

ClustMRF vs SDM per-query MAP:
  Wins   : 35
  Losses : 45
  Ties   : 2

Top-5 gains:
    qid  SDM_MAP  ClustMRF_MAP  Δ MAP
3092205   0.2561        0.6226 0.3665
2034768   0.0988        0.2612 0.1624
2033228   0.6589        0.8182 0.1593
3010623   0.2120        0.3692 0.1572
3100289   0.2558        0.3313 0.0755

Top-5 losses:
    qid  SDM_MAP  ClustMRF_MAP   Δ MAP
2040064   0.6036        0.4048 -0.1988
3100164   0.4789        0.3403 -0.1386
2047929   0.8047        0.6744 -0.1303
2046027   0.3439        0.2571 -0.0868
2003061   0.2828        0.2009 -0.0819


## 11  Ablation: Feature Groups

Compare three feature subsets to see which clique types drive the improvement:

- **qsim-only** — lQD + lQC only (geo/min/max/stdv-qsim), equal weights
- **qsim + dsim** — adds lC P_dsim features
- **full** — all 13 features (default weights)

In [26]:
ablation_rows = []

W4  = 1 / 4    # equal weight for 4 qsim features
W7  = 1 / 7    # equal weight for 7 qsim+dsim features

feature_configs = {
    'qsim-only': dict(
        w_geo_qsim=W4, w_stdv_qsim=W4, w_max_qsim=W4, w_min_qsim=W4,
        w_geo_dsim=0, w_min_dsim=0, w_max_dsim=0,
        w_geo_entropy=0, w_min_entropy=0, w_max_entropy=0,
        w_geo_icompress=0, w_min_icompress=0, w_max_icompress=0,
    ),
    'qsim+dsim': dict(
        w_geo_qsim=W7, w_stdv_qsim=W7, w_max_qsim=W7, w_min_qsim=W7,
        w_geo_dsim=W7, w_min_dsim=W7, w_max_dsim=W7,
        w_geo_entropy=0, w_min_entropy=0, w_max_entropy=0,
        w_geo_icompress=0, w_min_icompress=0, w_max_icompress=0,
    ),
    'full (default weights)': {},   # use constructor defaults
}

for name, overrides in feature_configs.items():
    cmrf = ClustMRF(index=index, k=5, n_docs=250, **overrides)
    run  = cmrf.transform(sdm_run)   # all ablations re-rank the SDM list
    ev   = run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, ev)
    row  = {'System': f'ClustMRF [{name}]'}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    ablation_rows.append(row)
    print(f'  {name}: done')

ab_df = pd.DataFrame(ablation_rows).set_index('System')
print('\n=== Ablation: feature groups ===')
print(ab_df.to_string())

  qsim-only: done
  qsim+dsim: done
  full (default weights): done

=== Ablation: feature groups ===
                                    AP@50     P@5  nDCG@5  nDCG@10  nDCG@100
System                                                                      
ClustMRF [qsim-only]               0.2612  0.5317  0.4425   0.4486    0.5570
ClustMRF [qsim+dsim]               0.2662  0.5512  0.4490   0.4587    0.5594
ClustMRF [full (default weights)]  0.2628  0.5585  0.4553   0.4598    0.5619


## 12  Ablation: Cluster Size k

k controls how many nearest neighbors form each document's cluster.
Larger k = smoother cluster LM (more documents pooled); smaller k = more focused.

In [ ]:
k_rows = []

for k_val in [3, 5, 10, 20]:
    cmrf_k = ClustMRF(index=index, k=k_val, n_docs=100)
    run_k  = cmrf_k.transform(sdm_run)   # re-rank the SDM list at each k
    ev_k   = run_k.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    agg_k  = ir_measures.calc_aggregate(MEASURES, qrels_df, ev_k)
    row = {'System': f'ClustMRF k={k_val}'}
    for m in MEASURES:
        row[str(m)] = round(float(agg_k[m]), 4)
    k_rows.append(row)
    print(f'  k={k_val}: done')

k_df = pd.DataFrame(k_rows).set_index('System')
print('\n=== Ablation: cluster size k ===')
print(k_df.to_string())

## 13  Save Runs (TREC format)

Write the BM25 and ClustMRF runs in standard TREC format for submission or
further analysis with `trec_eval`.

In [ ]:
# TREC run files — saved inline when each run cell executes;
# re-save here in case cells were skipped after loading from cache.
save_trec_run(bm25_run,       RUNS_DIR / 'bm25.txt',     'BM25_Terrier')
save_trec_run(sdm_run,        RUNS_DIR / 'sdm.txt',      'SDM_DirichletLM')
save_trec_run(clustmrf_run,   RUNS_DIR / 'clustmrf.txt', 'ClustMRF_SDM')

## 14  Interpolation: ClustMRF + BM25 / SDM

Linear score interpolation between ClustMRF and a base retrieval model:

```
score(d, q) = α · score_ClustMRF(d, q)  +  (1 − α) · score_base(d, q)
```

Scores are **min-max normalised per query** to [0, 1] before blending so the two systems' score scales don't dominate the mix.  Only documents appearing in **both** runs are retained (intersection).

α = 1.0 → pure ClustMRF; α = 0.0 → pure base model.

In [ ]:
def minmax_norm(run_df: pd.DataFrame) -> pd.DataFrame:
    """Min-max normalise scores to [0, 1] per query."""
    run_df = run_df.copy()
    grp = run_df.groupby('qid')['score']
    lo  = grp.transform('min')
    hi  = grp.transform('max')
    run_df['score'] = (run_df['score'] - lo) / (hi - lo).replace(0, 1.0)
    return run_df


def interpolate_runs(run_a: pd.DataFrame, run_b: pd.DataFrame,
                     alpha: float) -> pd.DataFrame:
    """alpha * run_a + (1-alpha) * run_b; both runs should be normalised first."""
    a = run_a[['qid', 'docno', 'score']].rename(columns={'score': 'score_a'})
    b = run_b[['qid', 'docno', 'score']].rename(columns={'score': 'score_b'})
    merged = pd.merge(a, b, on=['qid', 'docno'])
    merged['score'] = alpha * merged['score_a'] + (1.0 - alpha) * merged['score_b']
    merged = (merged[['qid', 'docno', 'score']]
              .sort_values(['qid', 'score'], ascending=[True, False]))
    merged['rank'] = merged.groupby('qid').cumcount() + 1
    return merged.reset_index(drop=True)


# Normalise once; reuse across all alpha values
bm25_norm     = minmax_norm(bm25_run)
sdm_norm      = minmax_norm(sdm_run)
clustmrf_norm = minmax_norm(clustmrf_run)

ALPHAS = [0.1, 0.25, 0.5, 0.75, 0.9]

interp_rows = []

# Reference baselines
for name, agg in [('BM25', bm25_agg), ('SDM', sdm_agg), ('ClustMRF', clustmrf_agg)]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    interp_rows.append(row)

interp_rows.append({'System': '---', **{str(m): '---' for m in MEASURES}})

for alpha in ALPHAS:
    # ClustMRF + BM25
    run_cb = interpolate_runs(clustmrf_norm, bm25_norm, alpha)
    ev_cb  = run_cb.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    agg_cb = ir_measures.calc_aggregate(MEASURES, qrels_df, ev_cb)
    row = {'System': f'ClustMRF+BM25  α={alpha}'}
    for m in MEASURES:
        row[str(m)] = round(float(agg_cb[m]), 4)
    interp_rows.append(row)

    # ClustMRF + SDM
    run_cs = interpolate_runs(clustmrf_norm, sdm_norm, alpha)
    ev_cs  = run_cs.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    agg_cs = ir_measures.calc_aggregate(MEASURES, qrels_df, ev_cs)
    row = {'System': f'ClustMRF+SDM   α={alpha}'}
    for m in MEASURES:
        row[str(m)] = round(float(agg_cs[m]), 4)
    interp_rows.append(row)

interp_df = pd.DataFrame(interp_rows).set_index('System')
print('=== Interpolation: ClustMRF + BM25 / SDM ===')
print('(α = weight on ClustMRF; 1−α = weight on base model)\n')
print(interp_df.to_string())

## 15  Dense Re-ranking: E5 and BGE

Re-rank the SDM top-100 candidates using two bi-encoder models:

| Model | Size | Query prefix | Doc prefix |
|-------|------|-------------|------------|
| `intfloat/e5-base-v2` | 109M | `query: ` | `passage: ` |
| `BAAI/bge-base-en-v1.5` | 109M | `Represent this sentence: ` | *(none)* |

Both use **mean pooling** over non-padding tokens followed by **L2 normalisation**; documents are scored by cosine similarity to the query vector.  Using the same SDM top-100 candidate pool as ClustMRF keeps the comparison fair.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODE_BATCH = 64

def mean_pool_encode(model, tokenizer, texts: list[str]) -> np.ndarray:
    """Mean-pool over non-padding tokens + L2 normalise → (len(texts), d)."""
    enc = tokenizer(texts, padding=True, truncation=True, max_length=512,
                    return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        out = model(**enc)
    mask   = enc['attention_mask'].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return F.normalize(pooled, dim=-1).cpu().float().numpy()


def dense_rerank(run_df: pd.DataFrame, model, tokenizer,
                 query_col: str = 'query_0',
                 query_prefix: str = '',
                 doc_prefix: str = '') -> pd.DataFrame:
    """Score each candidate by cosine sim to the query vector and re-rank."""
    results = []
    for qid, group in tqdm(run_df.groupby('qid'), desc='queries', leave=True):
        query = query_prefix + str(group[query_col].iloc[0])
        docs  = [doc_prefix + t for t in group['text'].fillna('').tolist()]

        d_vecs = np.vstack([
            mean_pool_encode(model, tokenizer, docs[i:i + ENCODE_BATCH])
            for i in range(0, len(docs), ENCODE_BATCH)
        ])
        q_vec  = mean_pool_encode(model, tokenizer, [query])
        scores = (d_vecs @ q_vec.T).squeeze(-1)

        grp_out = group.copy()
        grp_out['score'] = scores
        grp_out = grp_out.sort_values('score', ascending=False)
        grp_out['rank'] = range(1, len(grp_out) + 1)
        results.append(grp_out)

    return pd.concat(results, ignore_index=True)


print(f'Device: {DEVICE}')

print('\nLoading intfloat/e5-base-v2…')
e5_tok   = AutoTokenizer.from_pretrained('intfloat/e5-base-v2')
e5_model = AutoModel.from_pretrained('intfloat/e5-base-v2').to(DEVICE).eval()

print('Loading BAAI/bge-base-en-v1.5…')
bge_tok   = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
bge_model = AutoModel.from_pretrained('BAAI/bge-base-en-v1.5').to(DEVICE).eval()

_e5_path  = RUNS_DIR / 'e5.txt'
_bge_path = RUNS_DIR / 'bge.txt'

if _e5_path.exists():
    print('\nLoading cached E5 run…')
    e5_run = load_trec_run(_e5_path)
else:
    print('\nE5 re-ranking…')
    e5_run = dense_rerank(sdm_run, e5_model, e5_tok,
                           query_prefix='query: ', doc_prefix='passage: ')
    save_trec_run(e5_run, _e5_path, 'E5_base_v2')

if _bge_path.exists():
    print('Loading cached BGE run…')
    bge_run = load_trec_run(_bge_path)
else:
    print('\nBGE re-ranking…')
    bge_run = dense_rerank(sdm_run, bge_model, bge_tok,
                            query_prefix='Represent this sentence: ')
    save_trec_run(bge_run, _bge_path, 'BGE_base_v15')

e5_eval  = e5_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
bge_eval = bge_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
e5_agg   = ir_measures.calc_aggregate(MEASURES, qrels_df, e5_eval)
bge_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, bge_eval)

dense_rows = []
for name, agg in [
    ('BM25',             bm25_agg),
    ('SDM',              sdm_agg),
    ('ClustMRF',         clustmrf_agg),
    ('E5-base-v2',       e5_agg),
    ('BGE-base-en-v1.5', bge_agg),
]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    dense_rows.append(row)

dense_df = pd.DataFrame(dense_rows).set_index('System')
print('\n=== Dense Re-ranking vs Sparse/ClustMRF (SDM top-100 candidates) ===')
print(dense_df.to_string())

### 15.1  E5 Cluster Re-ranking

Clusters the SDM top-100 documents into **k=5** groups using K-means on E5 embeddings.
Clusters are ordered by centroid–query cosine similarity; within each cluster passages are
ordered by individual passage–query cosine similarity.

In [ ]:
import importlib
import src.algorithms.e5_cluster_reranker as _e5cr_mod
importlib.reload(_e5cr_mod)
from src.algorithms.e5_cluster_reranker import E5ClusterReranker

e5_cluster = E5ClusterReranker(e5_model, e5_tok, k=5, mode='knn', device=DEVICE)

_e5c_path = RUNS_DIR / 'e5_cluster.txt'
if _e5c_path.exists():
    print('Loading cached E5-Cluster run…')
    e5_cluster_run = load_trec_run(_e5c_path)
else:
    print('E5-cluster re-ranking (knn=5)…')
    e5_cluster_run = e5_cluster.transform(sdm_run)
    save_trec_run(e5_cluster_run, _e5c_path, 'E5_Cluster_knn5')

e5_cluster_eval = e5_cluster_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
e5_cluster_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, e5_cluster_eval)

all_rows = []
for name, agg in [
    ('BM25',              bm25_agg),
    ('SDM',               sdm_agg),
    ('ClustMRF',          clustmrf_agg),
    ('E5-base-v2',        e5_agg),
    ('BGE-base-en-v1.5',  bge_agg),
    ('E5-Cluster knn=5',  e5_cluster_agg),
]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    all_rows.append(row)

all_df = pd.DataFrame(all_rows).set_index('System')
print('\n=== All Re-rankers (SDM top-100 candidates) ===')
print(all_df.to_string())

---

## Summary

| System | MAP@50 | P@5 | NDCG@10 |
|--------|--------|-----|---------|
| BM25 (Terrier) | *see §6* | | |
| SDM (Metzler & Croft 2005) | *see §6* | | |
| ClustMRF (re-ranks SDM) | *see §9* | | |

**Pipeline:** SDM retrieves top-100 with full proximity scoring (λ_T=0.85/λ_O=0.10/λ_U=0.05,
DirichletLM μ=2500). ClustMRF clusters each doc with its k=5 nearest neighbours (TF-IDF cosine),
scores clusters with 13 MRF features (lQD/lQC/lC cliques), then unrolls docs from
best→worst cluster in sim(Q,d) order.

**Key observations to look for:**
- SDM gain over BM25: proximity terms help on verbose/multi-concept queries
- ClustMRF gain over SDM: cluster coherence promotes topically focused groups early;
  lC features (entropy, icompress) reward dense informative docs independently of query
- Feature ablation (§11): qsim-only vs full — how much do lC features contribute?
- k ablation (§12): larger clusters smooth out noise but reduce focus